# Reaseguro en Seguros de Vida y Daños

## Quota Share, Excess of Loss y Stop Loss

El reaseguro es una herramienta fundamental para que las aseguradoras gestionen su exposición al riesgo. Este notebook demuestra tres estructuras comunes de reaseguro.

### Contenido
1. Introducción al reaseguro
2. Quota Share (Cuota Parte)
3. Excess of Loss (Exceso de Pérdida)
4. Stop Loss (Exceso de Siniestralidad)
5. Comparación de estructuras
6. Casos prácticos

In [ ]:
# Imports necesarios
from decimal import Decimal
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Imports de reaseguro
from mexican_insurance.reinsurance.quota_share import QuotaShare
from mexican_insurance.reinsurance.excess_of_loss import ExcessOfLoss
from mexican_insurance.reinsurance.stop_loss import StopLoss
from mexican_insurance.core.validators import Moneda

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Imports completados exitosamente")

## 1. Introducción al Reaseguro

El reaseguro permite a las aseguradoras:
- Transferir parte del riesgo a un reasegurador
- Proteger su capital ante eventos catastróficos
- Estabilizar resultados técnicos
- Aumentar capacidad de suscripción

### Tipos principales:
1. **Proporcional** (Quota Share): Se cede un % fijo de todo
2. **No Proporcional** (XL, Stop Loss): Se cede solo lo que excede cierto límite

## 2. Quota Share (Cuota Parte)

En un contrato Quota Share, la aseguradora cede un porcentaje fijo de todas las primas y siniestros al reasegurador.

In [ ]:
# Crear contrato Quota Share: 40% cesión
qs_40 = QuotaShare(
    porcentaje_cesion=Decimal("0.40"),  # 40% al reasegurador
    comision_reaseguro=Decimal("0.25"),  # 25% de comisión sobre prima cedida
    moneda=Moneda.MXN
)

print("📋 CONTRATO QUOTA SHARE")
print("="*60)
print(f"Cesión: {qs_40.porcentaje_cesion * 100}%")
print(f"Retención: {qs_40.porcentaje_retencion * 100}%")
print(f"Comisión de reaseguro: {qs_40.comision_reaseguro * 100}%")

In [ ]:
# Caso práctico: Cartera de seguros de vida
prima_total = Decimal("10000000")  # 10 millones de prima anual
siniestros_total = Decimal("6500000")  # 6.5 millones en siniestros

# Calcular resultado con reaseguro
resultado_qs = qs_40.calcular_resultado(
    prima_directa=prima_total,
    siniestros=siniestros_total
)

print("\n💰 RESULTADO QUOTA SHARE 40%")
print("="*60)
print(f"\nPrimas:")
print(f"  Prima Directa Total:     ${resultado_qs['prima_directa']:>15,.2f}")
print(f"  Prima Cedida (40%):      ${resultado_qs['prima_cedida']:>15,.2f}")
print(f"  Prima Retenida (60%):    ${resultado_qs['prima_retenida']:>15,.2f}")
print(f"  Comisión Recibida (25%): ${resultado_qs['comision_recibida']:>15,.2f}")

print(f"\nSiniestros:")
print(f"  Siniestros Totales:      ${resultado_qs['siniestros']:>15,.2f}")
print(f"  Siniestros Cedidos:      ${resultado_qs['siniestros_cedidos']:>15,.2f}")
print(f"  Siniestros Retenidos:    ${resultado_qs['siniestros_retenidos']:>15,.2f}")

print(f"\nResultado Neto:")
print(f"  Resultado Aseguradora:   ${resultado_qs['resultado_neto_aseguradora']:>15,.2f}")

ratio_sin = (siniestros_total / prima_total) * 100
print(f"\n📊 Ratio de siniestralidad: {ratio_sin:.1f}%")

In [ ]:
# Visualización del Quota Share
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Distribución de Primas
primas_data = [
    float(resultado_qs['prima_retenida']),
    float(resultado_qs['prima_cedida'])
]
labels_primas = ['Retenida (60%)', 'Cedida (40%)']
colors = ['#3498db', '#e74c3c']

wedges1, texts1, autotexts1 = ax1.pie(primas_data, labels=labels_primas, autopct='%1.1f%%',
                                        startangle=90, colors=colors, textprops={'fontsize': 11})
ax1.set_title('Distribución de Primas\nQuota Share 40%', fontsize=13, fontweight='bold')

# Distribución de Siniestros
siniestros_data = [
    float(resultado_qs['siniestros_retenidos']),
    float(resultado_qs['siniestros_cedidos'])
]
labels_siniestros = ['Retenidos (60%)', 'Cedidos (40%)']

wedges2, texts2, autotexts2 = ax2.pie(siniestros_data, labels=labels_siniestros, autopct='%1.1f%%',
                                        startangle=90, colors=colors, textprops={'fontsize': 11})
ax2.set_title('Distribución de Siniestros\nQuota Share 40%', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Excess of Loss (Exceso de Pérdida)

El XL protege por siniestro individual. El reasegurador paga lo que excede una prioridad hasta un límite.

In [ ]:
# Crear contrato Excess of Loss: 4M xs 1M
# Significa: Reasegurador paga hasta 4M de lo que exceda 1M por siniestro
xl_contrato = ExcessOfLoss(
    prioridad=Decimal("1000000"),     # Aseguradora retiene primeros $1M
    limite=Decimal("4000000"),        # Reasegurador cubre hasta $4M más
    prima_reaseguro=Decimal("500000"),  # Prima anual del contrato
    moneda=Moneda.MXN
)

print("📋 CONTRATO EXCESS OF LOSS")
print("="*60)
print(f"Prioridad (retención): ${xl_contrato.prioridad:,.0f}")
print(f"Límite (cobertura XL): ${xl_contrato.limite:,.0f}")
print(f"Capacidad total:       ${xl_contrato.prioridad + xl_contrato.limite:,.0f}")
print(f"Prima de reaseguro:    ${xl_contrato.prima_reaseguro:,.0f}")

In [ ]:
# Simular diferentes siniestros
siniestros_xl = [
    Decimal("500000"),    # Bajo la prioridad
    Decimal("1500000"),   # Parcialmente cubierto
    Decimal("3000000"),   # Totalmente en la capa XL
    Decimal("6000000"),   # Excede el límite XL
]

print("\n💰 ANÁLISIS DE SINIESTROS INDIVIDUALES")
print("="*80)

resultados_xl = []
for i, monto in enumerate(siniestros_xl, 1):
    resultado = xl_contrato.aplicar_siniestro(monto)
    resultados_xl.append(resultado)
    
    print(f"\nSiniestro {i}: ${monto:,.0f}")
    print(f"  - Aseguradora paga: ${resultado['monto_aseguradora']:>12,.0f}")
    print(f"  - Reasegurador paga: ${resultado['monto_reasegurador']:>12,.0f}")
    print(f"  - Excedente: ${resultado['excedente']:>12,.0f}")
    
    pct_cedido = (resultado['monto_reasegurador'] / monto * 100) if monto > 0 else 0
    print(f"  - % cedido al reasegurador: {pct_cedido:.1f}%")

In [ ]:
# Visualización del XL
fig, ax = plt.subplots(figsize=(12, 6))

siniestros_labels = ['$500K', '$1.5M', '$3M', '$6M']
x_pos = np.arange(len(siniestros_labels))
width = 0.35

# Preparar datos
monto_aseg = [float(r['monto_aseguradora']) for r in resultados_xl]
monto_reas = [float(r['monto_reasegurador']) for r in resultados_xl]
excedente = [float(r['excedente']) for r in resultados_xl]

# Barras apiladas
p1 = ax.bar(x_pos, monto_aseg, width, label='Aseguradora (retención)', color='#3498db', alpha=0.8)
p2 = ax.bar(x_pos, monto_reas, width, bottom=monto_aseg, label='Reasegurador (XL)', 
            color='#2ecc71', alpha=0.8)
p3 = ax.bar(x_pos, excedente, width, 
            bottom=[m1+m2 for m1, m2 in zip(monto_aseg, monto_reas)],
            label='Excedente (sin cobertura)', color='#e74c3c', alpha=0.8)

ax.set_ylabel('Monto (MXN)', fontsize=12)
ax.set_title('Distribución de Siniestros - Excess of Loss\nPrioridad: $1M | Límite: $4M', 
             fontsize=13, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(siniestros_labels)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

# Líneas de referencia
ax.axhline(y=1000000, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Prioridad')
ax.axhline(y=5000000, color='orange', linestyle='--', linewidth=2, alpha=0.5, 
           label='Límite total (Prioridad + XL)')

plt.tight_layout()
plt.show()

## 4. Stop Loss (Exceso de Siniestralidad)

El Stop Loss protege el resultado agregado de la cartera. Cubre cuando la siniestralidad excede cierto ratio.

In [ ]:
# Crear contrato Stop Loss: 90% xs 75%
# Significa: Si siniestralidad excede 75%, reasegurador paga hasta llegar a 90%
sl_contrato = StopLoss(
    ratio_attachment=Decimal("0.75"),  # 75% - punto de activación
    ratio_limite=Decimal("0.90"),      # 90% - límite de cobertura
    prima_sujeta=Decimal("10000000"),  # Base de cálculo (prima anual)
    prima_reaseguro=Decimal("300000"), # Prima del contrato
    moneda=Moneda.MXN
)

print("📋 CONTRATO STOP LOSS")
print("="*60)
print(f"Attachment point: {sl_contrato.ratio_attachment * 100}%")
print(f"Límite: {sl_contrato.ratio_limite * 100}%")
print(f"Capa de cobertura: {(sl_contrato.ratio_limite - sl_contrato.ratio_attachment) * 100}%")
print(f"Prima sujeta: ${sl_contrato.prima_sujeta:,.0f}")
print(f"Prima de reaseguro: ${sl_contrato.prima_reaseguro:,.0f}")

In [ ]:
# Simular diferentes escenarios de siniestralidad
escenarios = [
    ("Favorable", Decimal("6000000"), 60),   # 60% siniestralidad
    ("Normal", Decimal("7000000"), 70),      # 70% siniestralidad
    ("Alto", Decimal("8000000"), 80),        # 80% siniestralidad - activa SL
    ("Severo", Decimal("9500000"), 95),      # 95% siniestralidad - excede SL
]

print("\n💰 ANÁLISIS DE ESCENARIOS")
print("="*90)

resultados_sl = []
for nombre, siniestros, ratio in escenarios:
    resultado = sl_contrato.calcular_resultado(
        prima_devengada=sl_contrato.prima_sujeta,
        siniestros_incurridos=siniestros
    )
    resultados_sl.append((nombre, resultado))
    
    print(f"\nEscenario: {nombre} ({ratio}% siniestralidad)")
    print(f"  - Prima devengada:           ${resultado['prima_devengada']:>12,.0f}")
    print(f"  - Siniestros incurridos:     ${resultado['siniestros_incurridos']:>12,.0f}")
    print(f"  - Ratio siniestralidad:      {resultado['ratio_siniestralidad']*100:>12.1f}%")
    print(f"  - Recuperación reasegurador: ${resultado['recuperacion_reasegurador']:>12,.0f}")
    print(f"  - Siniestros netos:          ${resultado['siniestros_netos_aseguradora']:>12,.0f}")
    print(f"  - Ratio neto aseguradora:    {resultado['ratio_neto_aseguradora']*100:>12.1f}%")

In [ ]:
# Visualización del Stop Loss
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfica 1: Comparación de ratios
nombres_esc = [e[0] for e in escenarios]
ratios_brutos = [e[1] * 100 for _, e in resultados_sl]
ratios_netos = [float(e[1]['ratio_neto_aseguradora'] * 100) for e in resultados_sl]

x = np.arange(len(nombres_esc))
width = 0.35

bars1 = ax1.bar(x - width/2, ratios_brutos, width, label='Sin Stop Loss', 
                color='#e74c3c', alpha=0.7)
bars2 = ax1.bar(x + width/2, ratios_netos, width, label='Con Stop Loss', 
                color='#2ecc71', alpha=0.7)

ax1.set_ylabel('Ratio de Siniestralidad (%)', fontsize=12)
ax1.set_title('Impacto del Stop Loss en Siniestralidad', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(nombres_esc)
ax1.legend(fontsize=11)
ax1.grid(axis='y', alpha=0.3)
ax1.axhline(y=75, color='orange', linestyle='--', linewidth=2, label='Attachment (75%)')
ax1.axhline(y=90, color='red', linestyle='--', linewidth=2, label='Límite (90%)')

# Agregar valores sobre barras
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.0f}%', ha='center', va='bottom', fontsize=10)

# Gráfica 2: Recuperaciones del reasegurador
recuperaciones = [float(e[1]['recuperacion_reasegurador']) for e in resultados_sl]
colors_rec = ['#95a5a6' if r == 0 else '#2ecc71' for r in recuperaciones]

bars3 = ax2.bar(nombres_esc, recuperaciones, color=colors_rec, alpha=0.7, edgecolor='black')
ax2.set_ylabel('Recuperación (MXN)', fontsize=12)
ax2.set_title('Recuperaciones del Reasegurador\npor Escenario', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

for bar in bars3:
    height = bar.get_height()
    if height > 0:
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'${height/1e6:.2f}M', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Comparación de Estructuras

Comparemos cómo cada tipo de reaseguro afecta el resultado de la aseguradora.

In [ ]:
# Simular un año con múltiples siniestros
siniestros_año = [
    Decimal("500000"),
    Decimal("1200000"),
    Decimal("800000"),
    Decimal("3500000"),  # Siniestro grande
    Decimal("600000"),
    Decimal("2000000"),
    Decimal("750000"),
]

total_siniestros = sum(siniestros_año)
prima_anual = Decimal("10000000")

print("📊 COMPARACIÓN DE ESTRATEGIAS DE REASEGURO")
print("="*70)
print(f"Prima anual: ${prima_anual:,.0f}")
print(f"Total siniestros: ${total_siniestros:,.0f}")
print(f"Ratio siniestralidad bruto: {(total_siniestros/prima_anual)*100:.1f}%")
print(f"Cantidad de siniestros: {len(siniestros_año)}")

# Sin reaseguro
resultado_sin_reas = prima_anual - total_siniestros

# Con Quota Share 40%
res_qs_comp = qs_40.calcular_resultado(prima_anual, total_siniestros)
costo_qs = res_qs_comp['prima_cedida'] - res_qs_comp['comision_recibida']
resultado_qs_comp = res_qs_comp['resultado_neto_aseguradora']

# Con Excess of Loss
total_recuperado_xl = sum([xl_contrato.aplicar_siniestro(s)['monto_reasegurador'] 
                           for s in siniestros_año])
resultado_xl_comp = prima_anual - total_siniestros + total_recuperado_xl - xl_contrato.prima_reaseguro

# Con Stop Loss
res_sl_comp = sl_contrato.calcular_resultado(prima_anual, total_siniestros)
resultado_sl_comp = (prima_anual - res_sl_comp['siniestros_netos_aseguradora'] - 
                     sl_contrato.prima_reaseguro)

print("\n💰 RESULTADOS NETOS:")
print("="*70)
print(f"Sin reaseguro:         ${resultado_sin_reas:>15,.0f}")
print(f"Con Quota Share 40%:   ${resultado_qs_comp:>15,.0f}")
print(f"Con Excess of Loss:    ${resultado_xl_comp:>15,.0f}")
print(f"Con Stop Loss:         ${resultado_sl_comp:>15,.0f}")

In [ ]:
# Visualización comparativa final
fig, ax = plt.subplots(figsize=(12, 6))

estrategias = ['Sin Reaseguro', 'Quota Share\n40%', 'Excess of Loss\n4M xs 1M', 'Stop Loss\n90% xs 75%']
resultados = [
    float(resultado_sin_reas),
    float(resultado_qs_comp),
    float(resultado_xl_comp),
    float(resultado_sl_comp)
]

colors_comp = ['#e74c3c' if r < 0 else '#2ecc71' for r in resultados]
bars = ax.bar(estrategias, resultados, color=colors_comp, alpha=0.7, edgecolor='black', linewidth=2)

ax.set_ylabel('Resultado Neto (MXN)', fontsize=12)
ax.set_title('Comparación de Estrategias de Reaseguro\nPrima: $10M | Siniestros: ${:.1f}M'.format(
    float(total_siniestros)/1e6), fontsize=13, fontweight='bold')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.grid(axis='y', alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

# Agregar valores
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'${height/1e6:.2f}M',
            ha='center', va='bottom' if height > 0 else 'top', 
            fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Análisis comparativo completado")

## Conclusiones

### Quota Share
- **Ventajas**: Reduce exposición uniformemente, genera comisión de reaseguro
- **Desventajas**: Cede también los buenos resultados
- **Mejor para**: Aseguradoras nuevas, diversificación general

### Excess of Loss
- **Ventajas**: Protege contra siniestros grandes, retiene resultados normales
- **Desventajas**: Costo fijo (prima), no protege frecuencia
- **Mejor para**: Protección catastrófica, estabilizar volatilidad

### Stop Loss
- **Ventajas**: Protege el resultado global, limita pérdidas agregadas
- **Desventajas**: Costo alto, activación solo en escenarios adversos
- **Mejor para**: Protección de solvencia, carteras con alta variabilidad

### Próximos Pasos

Continúa con:
- **03_reservas_tecnicas.ipynb**: Chain Ladder, Bornhuetter-Ferguson, Bootstrap
- **04_cumplimiento_cnsf.ipynb**: RCS y cumplimiento regulatorio